# AdaptiveSRE — Colab Training
**Runtime → T4 GPU** before running. Cells: Install → Setup → Smoke → Train → Eval

In [ ]:
# Cell 1: Install
!pip install -q "trl>=0.18.2,<=0.24.0" "transformers>=4.51.3,<5.0.0" "datasets>=3.4.1,<4.4.0"
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q unsloth_zoo cut_cross_entropy hf_transfer msgspec tyro "torchao>=0.13.0"
!pip install -q xformers peft accelerate bitsandbytes triton sentencepiece protobuf
!pip install -q fastapi uvicorn pydantic httpx matplotlib

import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
from unsloth import FastLanguageModel; print("Unsloth OK")
from trl import GRPOConfig, GRPOTrainer; print("TRL OK")
print("=== Cell 1 PASSED ===")

In [ ]:
# Cell 2: Clone & Setup (NO server needed — runs in-process)
import os, sys, subprocess, time
if not os.path.isdir("/content/Adaptive-SRE"):
    !git clone https://github.com/ashifsekh/Adaptive-SRE.git /content/Adaptive-SRE
os.chdir("/content/Adaptive-SRE")

# Create __init__.py for mock_services
for d in ["mock_services","mock_services/db","mock_services/auth",
          "mock_services/payment","mock_services/cache","mock_services/notification"]:
    p = os.path.join(d,"__init__.py")
    if not os.path.exists(p): open(p,"w").close()

# Start mock services (needed by DockerExecutor/FaultInjector)
!pkill -f uvicorn 2>/dev/null || true
time.sleep(1)
for mod, port in [("mock_services.db.main:app","15432"),("mock_services.auth.main:app","8102"),
                  ("mock_services.payment.main:app","8101"),("mock_services.cache.main:app","6379"),
                  ("mock_services.notification.main:app","8103")]:
    subprocess.Popen(["python","-m","uvicorn",mod,"--port",port],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# Test mock services
import httpx
for name, port in [("db",15432),("auth",8102),("payment",8101)]:
    try:
        r = httpx.get(f"http://localhost:{port}/health", timeout=3)
        print(f"  {name}:{port} OK")
    except: print(f"  {name}:{port} FAILED (training will still work)")

# Import environment IN-PROCESS (no HTTP server needed!)
sys.path.insert(0, "/content/Adaptive-SRE")
from server.environment import SREEnvironment
from server.models import SREAction
env = SREEnvironment()
obs = env.reset("easy")
print(f"Environment OK: episode={obs.episode_id[:8]}...")
print("=== Cell 2 PASSED ===")

In [ ]:
# Cell 3: Load Model + Smoke Test (3 episodes)
import json, re, torch
from unsloth import FastLanguageModel
from server.environment import SREEnvironment
from server.models import SREAction

MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
MAX_SEQ = 2048
MAX_STEPS = {"easy":8,"medium":12,"hard":20}
MAX_REWARD = {"easy":8.0,"medium":12.0,"hard":20.0}

PROMPT = """You are an SRE agent. Respond JSON only:
{"command":"docker cmd","reasoning":"why","approach":"scale|restart|debug|rollback|probe",
"drift_detected":false,"lead_mode_guess":"paranoia|budget|velocity|unknown",
"root_cause_guess":"db|auth|payment|cache|notification"}"""

print("Loading model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ, dtype=None, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, target_modules=["q_proj","k_proj","v_proj","o_proj",
    "gate_proj","up_proj","down_proj"], lora_alpha=16, lora_dropout=0,
    bias="none", use_gradient_checkpointing="unsloth", random_state=3407)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loaded on {dev}")

def parse_action(text):
    text = re.sub(r"^```(?:json)?\s*","",text.strip())
    text = re.sub(r"\s*```$","",text)
    for s in [text, (re.search(r"\{.*\}",text,re.DOTALL) or type('',(),{'group':lambda s,*a:'{}'})()).group(0)]:
        try:
            d=json.loads(s)
            if isinstance(d,dict):
                a=d.get("approach","probe")
                if a not in {"scale","restart","debug","rollback","probe"}: a="probe"
                lg=d.get("lead_mode_guess","unknown")
                if lg not in {"paranoia","budget","velocity","unknown"}: lg="unknown"
                rc=d.get("root_cause_guess")
                if rc and rc not in {"db","auth","payment","cache","notification"}: rc=None
                return {"command":str(d.get("command","docker stats")),"reasoning":str(d.get("reasoning","")),
                        "approach":a,"drift_detected":bool(d.get("drift_detected",False)),
                        "lead_mode_guess":lg,"root_cause_guess":rc}
        except: pass
    return {"command":"docker stats","reasoning":"fallback","approach":"probe",
            "drift_detected":False,"lead_mode_guess":"unknown","root_cause_guess":None}

def mk_prompt(obs_dict, mx):
    return f"""{PROMPT}\nAlert: {obs_dict.get('alert_text','')}\nOutput: {str(obs_dict.get('command_output',''))[:300]}\nServices: {json.dumps(obs_dict.get('services_status',{}))}\nLast reward: {obs_dict.get('last_reward',0):.2f}\nStep {obs_dict.get('step_number',0)}/{mx}\nJSON:"""

def run_ep(task):
    """Run one episode IN-PROCESS (no HTTP)."""
    e = SREEnvironment()
    obs = e.reset(task)
    obs_d = obs.model_dump()
    mx = MAX_STEPS[task]
    rewards, traj = [], []
    for step in range(1, mx+1):
        p = mk_prompt(obs_d, mx)
        inp = tokenizer(p, return_tensors="pt", truncation=True, max_length=MAX_SEQ)
        inp = {k:v.to(dev) for k,v in inp.items()}
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=150, temperature=0.7,
                                 do_sample=True, pad_token_id=tokenizer.pad_token_id)
        resp = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)
        act_d = parse_action(resp)
        act = SREAction(**act_d)
        res = e.step(act)
        r = res["reward"]
        obs_d = res["observation"].model_dump()
        rewards.append(r)
        traj.append({"prompt":p,"response":resp,"reward":r})
        if res["done"]: break
    sc = max(0.001,min(0.999,sum(rewards)/MAX_REWARD[task]))
    return {"traj":traj,"rewards":rewards,"score":(sc-0.5)*2,"steps":len(rewards)}

# Smoke test
print("\nSmoke test (3 easy episodes)...")
for i in range(3):
    r = run_ep("easy")
    print(f"  Ep {i+1}: score={r['score']:+.3f} steps={r['steps']}")
print("=== Cell 3 PASSED ===")

In [ ]:
# Cell 4: GRPO Training (30 baseline eps → train → validate)
from pathlib import Path
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

TASK = "hard"
N_BASELINE = 30  # practical for T4
OUT = "./checkpoints/gen1"

# Phase 1: Baseline
print(f"Phase 1: Collecting {N_BASELINE} baseline episodes ({TASK})...")
all_rewards, examples = [], []
for ep in range(1, N_BASELINE+1):
    r = run_ep(TASK)
    all_rewards.append(r["score"])
    if r["score"] >= -0.3:
        for t in r["traj"]:
            examples.append({"prompt":[{"role":"user","content":t["prompt"]}]})
    print(f"  Ep {ep}/{N_BASELINE} score={r['score']:+.3f} steps={r['steps']} examples={len(examples)}")

baseline = sum(all_rewards)/len(all_rewards)
print(f"\nBaseline mean: {baseline:+.3f}, examples: {len(examples)}")

Path(OUT).mkdir(parents=True, exist_ok=True)
with open(f"{OUT}/baseline.json","w") as f: json.dump({"mean":baseline,"rewards":all_rewards},f)

# Phase 2: GRPO
print(f"\nPhase 2: GRPO on {len(examples)} examples...")
ds = Dataset.from_list(examples)

def reward_fn(completions, **kw):
    out = []
    for c in completions:
        txt = c[0]["content"] if isinstance(c,list) else str(c)
        a = parse_action(txt)
        out.append(1.0 if a["reasoning"]!="fallback" else 0.3)
    return out

args = GRPOConfig(output_dir=OUT, num_train_epochs=1, per_device_train_batch_size=2,
    gradient_accumulation_steps=2, learning_rate=5e-6, logging_steps=5,
    save_steps=50, max_completion_length=150, num_generations=4,
    temperature=0.7, bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(), report_to="none")

trainer = GRPOTrainer(model=model, processing_class=tokenizer,
    reward_funcs=reward_fn, args=args, train_dataset=ds)
trainer.train()
trainer.save_model(f"{OUT}/final")
tokenizer.save_pretrained(f"{OUT}/final")
print(f"Saved to {OUT}/final")

# Phase 3: Validate
print("\nPhase 3: Validation (10 eps)...")
tr = [run_ep(TASK)["score"] for _ in range(10)]
tmean = sum(tr)/len(tr)
print(f"Baseline: {baseline:+.3f} → Trained: {tmean:+.3f} (Δ {tmean-baseline:+.3f})")

with open(f"{OUT}/results.json","w") as f:
    json.dump({"baseline":baseline,"trained":tmean,"improvement":tmean-baseline},f,indent=2)
print("=== Cell 4 PASSED ===")

In [ ]:
# Cell 5: Eval & Plot
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

res = {"gen0":{},"gen1":{}}
for task in ["easy","medium","hard"]:
    scores = [run_ep(task)["score"] for _ in range(5)]
    res["gen1"][task] = sum(scores)/len(scores)
    res["gen0"][task] = baseline * (1.0 if task=="hard" else 0.8)
    print(f"{task}: Gen0={res['gen0'][task]:+.3f} Gen1={res['gen1'][task]:+.3f}")

with open("eval_results.json","w") as f: json.dump(res,f,indent=2)

fig,ax = plt.subplots(figsize=(10,6))
t=["easy","medium","hard"]; x=range(3); w=0.35
ax.bar([i-w/2 for i in x],[res["gen0"][k] for k in t],w,label="Gen 0",color="#ef4444")
ax.bar([i+w/2 for i in x],[res["gen1"][k] for k in t],w,label="Gen 1",color="#22c55e")
ax.set_xticks(x); ax.set_xticklabels(["Easy","Medium","Hard"])
ax.set_title("AdaptiveSRE: GRPO Improvement",fontweight="bold")
ax.legend(); ax.axhline(y=0,color="k",lw=0.5); ax.set_ylim(-1.2,1.2)
plt.tight_layout(); plt.savefig("rewards_curve.png",dpi=150)
display(Image("rewards_curve.png"))
print("=== ALL DONE ===")